# JIN AI — Phase 17: continued pretraining (tự dò input, chạy tuần tự)

Notebook này tự tìm dataset bạn đã gắn, tự chọn corpus, và **dừng ngay** nếu thiếu
checkpoint Phase 14 hoặc nếu dữ liệu không phải v7. Không có đường nào dẫn tới
train từ random weights hoặc train lại corpus cũ.

Chạy từ trên xuống. Stage A/B/C chỉ mất vài phút; Stage D (ingest + verify +
tokenize) lâu nhất; Stage G là train thật.

Dataset cần gắn: một dataset chứa `best.pt`, thư mục repo (hoặc zip repo), và các
file `*.parquet` của Wikipedia — ví dụ `jin-ai-phase17-inputs`.

Settings: **Accelerator = GPU T4 x2 / P100**, **Persistence = Files**.
Internet không bắt buộc.

## 0. Dò input và cấu hình

Không cần điền path. Cell này tìm `train_gpt2.py`, `best.pt`, và thư mục parquet
dưới `/kaggle/input`, rồi in ra đúng những gì nó tìm thấy.

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/repo"
OUT = "/kaggle/working/phase17_output"
os.makedirs(OUT, exist_ok=True)

def _first(pattern, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
    return hits[0] if hits else None

ckpt = _first("best.pt")
if not ckpt:
    raise SystemExit("KHÔNG thấy best.pt trong /kaggle/input — Add Input dataset chứa Phase 14 checkpoint.")
DS = os.path.dirname(ckpt)

parquet = _first("*.parquet")
if not parquet:
    raise SystemExit("KHÔNG thấy *.parquet trong /kaggle/input — Add Input dataset Wikipedia.")
PARQUET_DIR = os.path.dirname(parquet)
SOURCE_ID = os.path.basename(PARQUET_DIR.rstrip("/"))      # id nguồn cho ingestion

repo_entry = _first("train_gpt2.py")
repo_zip = None if repo_entry else _first("*.zip")
if not repo_entry and not repo_zip:
    raise SystemExit("KHÔNG thấy repo (train_gpt2.py hoặc .zip) trong /kaggle/input.")

CONFIG = {
    "repo_entry": repo_entry,
    "repo_zip": repo_zip,
    "dataset_dir": DS,
    "checkpoint_path": ckpt,
    "wikipedia_path": PARQUET_DIR,
    "source_id": SOURCE_ID,
    "license_map_path": "/kaggle/working/license_map.json",
    "phase17_ckpt_path": None,   # set ở Stage G2 nếu resume
    "eval_jsonl": None,         # set nếu bạn có file eval frozen Phase 16
    "out_dir": OUT,
}
print(json.dumps({k: v for k, v in CONFIG.items() if v}, indent=2, ensure_ascii=False))
print("parquet files:", len(glob.glob(os.path.join(PARQUET_DIR, "*.parquet"))))
print("checkpoint   :", round(os.path.getsize(ckpt) / 1e9, 2), "GB")

In [ ]:
# Copy code sang /kaggle/working (input là read-only). Nếu input là zip thì giải nén trước.
if CONFIG["repo_entry"]:
    src_repo = os.path.dirname(CONFIG["repo_entry"])
    if os.path.exists(WORK):
        shutil.rmtree(WORK)
    shutil.copytree(src_repo, WORK, ignore=shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc", "data", "checkpoints", "cache"))
else:
    os.makedirs(WORK, exist_ok=True)
    zipfile.ZipFile(CONFIG["repo_zip"]).extractall(WORK)
    inner = [d for d in os.listdir(WORK) if os.path.isfile(os.path.join(WORK, d, "train_gpt2.py"))]
    if inner:
        WORK = os.path.join(WORK, inner[0])

os.chdir(WORK)
assert os.path.exists("train_gpt2.py"), "repo không đúng cấu trúc"
CONFIG["repo_path"] = WORK
print("repo:", WORK)
print(sorted(os.listdir(WORK))[:10])
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)

## A. Preflight môi trường

Ghi GPU/VRAM/CUDA/torch/disk/HEAD/checkpoint SHA vào `kaggle_preflight.json`.
Cell này **assert** `host_kind == "kaggle"`: nếu chạy ngoài kernel Kaggle nó sẽ dừng,
để một artifact local không bị báo như bằng chứng Kaggle.

In [ ]:
subprocess.run([sys.executable, "scripts/kaggle_preflight.py",
                "--checkpoint", CONFIG["checkpoint_path"],
                "--out", f"{OUT}/kaggle_preflight.json"]).check_returncode()
preflight = json.load(open(f"{OUT}/kaggle_preflight.json", encoding="utf-8"))
assert preflight["host"]["host_kind"] == "kaggle", "không phải kernel Kaggle — dừng"
PARENT_SHA = preflight["checkpoint"]["sha256"]
print("parent sha256:", PARENT_SHA)
print("devices       :", preflight["torch"]["device_count"], "(trainer dùng device 0)")

## B/C. License map + xem nguồn

Ingestion từ chối nguồn không có licence. Licence dưới đây là bản ghi upstream của
chính dữ liệu này (Wikimedia, revision `b04c8d1c…`); nếu bạn dùng bộ khác, sửa
`LICENSE_TEXT` cho khớp dataset card trước khi chạy.

In [ ]:
LICENSE_TEXT = "CC-BY-SA-3.0 and GFDL as dataset card; review upstream terms"
SOURCE_URL = "https://huggingface.co/datasets/wikimedia/wikipedia"

license_map = {CONFIG["source_id"]: {
    "license": LICENSE_TEXT, "source_url": SOURCE_URL,
    "text_columns": ["text"], "language": "en",
}}
json.dump(license_map, open(CONFIG["license_map_path"], "w", encoding="utf-8"), indent=2)
print(json.dumps(license_map, indent=2, ensure_ascii=False))

subprocess.run([sys.executable, "scripts/ingest_corpus.py",
                "--input-root", CONFIG["wikipedia_path"], "--list"])

## C2. Sample 10k docs — đo trước khi ingest 11 GB

Từ sample suy ra bytes/token và chọn `MAX_DOCS` cho vừa dung lượng còn trống.
Không có bước này thì rất dễ hết disk giữa chừng.

In [ ]:
SAMPLE = f"{OUT}/curated_v7_sample"
subprocess.run([sys.executable, "scripts/ingest_corpus.py",
                "--input-root", CONFIG["wikipedia_path"],
                "--source", CONFIG["source_id"],
                "--license-map", CONFIG["license_map_path"],
                "--max-docs-per-source", "10000",
                "--out-root", SAMPLE]).check_returncode()

man = json.load(open(f"{SAMPLE}/raw_manifest.json", encoding="utf-8"))["sources"][0]
docs = man["documents"]; chars = man["chars"]
bytes_per_doc = max(1.0, chars / max(1, docs) * 1.15)      # JSONL + metadata overhead
tokens_per_doc = chars / 4.0                               # GPT-2 BPE ~4 chars/token
free_gb = shutil.disk_usage("/kaggle/working").free / 1e9

# Lượng dữ liệu cần: 400M token đủ cho mốc 100M train + eval + dư cho 300M sau này.
# Ingest 11 GB / 1M docs là không cần thiết và dễ hết disk, nên chặn bằng token target
# trước, rồi mới chặn bằng dung lượng trống.
TOKEN_TARGET = 400_000_000
docs_for_tokens = TOKEN_TARGET / tokens_per_doc
disk_cap = free_gb * 1e9 * 0.45 / bytes_per_doc      # chừa chỗ cho checkpoint (~3 GB/cái)
MAX_DOCS = int(min(1_000_000, max(20_000, min(docs_for_tokens, disk_cap))))
proj_gb = MAX_DOCS * bytes_per_doc / 1e9
proj_tokens = MAX_DOCS * tokens_per_doc
binding = "token target" if docs_for_tokens <= disk_cap else "disk free"

print(f"sample: {docs:,} docs | {chars/1e6:.1f}M chars | {man['docs_per_second']}/s")
print(f"per doc: {bytes_per_doc/1e3:.1f} kB | {tokens_per_doc:.0f} tokens")
print(f"free {free_gb:.0f} GB | docs needed for {TOKEN_TARGET/1e6:.0f}M tokens: {docs_for_tokens:,.0f}"
      f" | disk allows: {disk_cap:,.0f}")
print(f"MAX_DOCS={MAX_DOCS:,} (limited by {binding}) | JSONL ~{proj_gb:.1f} GB | ~{proj_tokens/1e6:.0f}M tokens")

## D1. Ingest → verify (dedup, split, licence, PII)

Chỉ ingest đúng một nguồn đã review. `--pii-policy exclude --license-policy enforce`
là bắt buộc. Đọc `reports/` chứ đừng tin exit code.

In [ ]:
CURATED = "data/curated_v7"
subprocess.run([sys.executable, "scripts/ingest_corpus.py",
                "--input-root", CONFIG["wikipedia_path"],
                "--source", CONFIG["source_id"],
                "--license-map", CONFIG["license_map_path"],
                "--max-docs-per-source", str(MAX_DOCS),
                "--out-root", CURATED]).check_returncode()

subprocess.run([sys.executable, "scripts/verify_corpus.py",
                "--in-root", CURATED, "--out-root", CURATED,
                "--pii-policy", "exclude", "--license-policy", "enforce"]).check_returncode()

for path in sorted(glob.glob(f"{CURATED}/reports/*.json")):
    report = json.load(open(path, encoding="utf-8"))
    print(os.path.basename(path), "->", json.dumps(report)[:300])

## D2. Tokenize → `data/gpt2_corpus_v7`

Mixture được suy ra từ **nguồn thực sự có** trong `curated_v7`, và in ra realized
mixture. Nếu chỉ có Wikipedia, nó nói thẳng là wiki-only — Phase 14 từng có
cosmopedia/prepared/seed, nên đây là khác biệt phải ghi nhận, không phải để lờ đi.

In [ ]:
raw = json.load(open(f"{CURATED}/raw_manifest.json", encoding="utf-8"))["sources"]
available = [s["source_id"] for s in raw if s.get("present") and s.get("documents")]
print("sources in curated_v7:", available)

DEFAULT_WEIGHTS = {"wikipedia_en_20231101": 0.65, "cosmopedia_v2_sample": 0.25,
                   "prepared_v4": 0.08, "project_seed": 0.02}
weights = {}
for sid in available:
    weights[sid] = DEFAULT_WEIGHTS.get(sid, 0.0)
if all(w == 0.0 for w in weights.values()):
    weights = {sid: 1.0 / len(available) for sid in available}
total = sum(weights.values())
weights = {k: v / total for k, v in weights.items()}
MIX = ",".join(f"{k}:{v:.2f}" for k, v in weights.items())
print("mix:", MIX)
if len(available) == 1:
    print("CẢNH BÁO: corpus v7 chỉ có 1 nguồn -> đây là wiki-only, khác mixture Phase 14.")

TOKENS = "data/gpt2_corpus_v7"
subprocess.run([sys.executable, "scripts/build_tokenized_corpus.py",
                "--splits", f"{CURATED}/splits", "--out", TOKENS,
                "--max-train-tokens", "300000000",
                "--mix", MIX]).check_returncode()
tok_man = json.load(open(f"{TOKENS}/manifest.json", encoding="utf-8"))
print(json.dumps({k: tok_man.get(k) for k in ("train", "val", "test")}, indent=2)[:600])

TRAIN_DATA = TOKENS
assert os.path.exists(f"{TRAIN_DATA}/manifest.json"), "tokenize thất bại"
print("TRAIN_DATA =", TRAIN_DATA)

## E. Smoke test từ Phase 14 (20 step, context 1024)

Điều kiện đúng: `best_val_loss` đầu tiên phải **< 6.0** (khoảng 4.7). Random init
của kiến trúc này ≈ 10.8. Cell sau kiểm tra bằng chính checkpoint, không đọc log.

In [ ]:
subprocess.run([sys.executable, "train_gpt2.py",
                "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                "--init-from", CONFIG["checkpoint_path"], "--continue-pretrain",
                "--out-dir", f"{OUT}/smoke", "--seq-len", "1024",
                "--batch-size", "4", "--grad-accum", "4",
                "--max-steps", "20", "--warmup", "2", "--lr", "1e-4",
                "--eval-interval", "10", "--eval-iters", "4",
                "--save-interval", "10", "--seed", "17"]).check_returncode()

In [ ]:
import torch
smoke = torch.load(f"{OUT}/smoke/latest.pt", map_location="cpu", weights_only=True, mmap=True)
smoke_man = json.load(open(f"{OUT}/smoke/latest.pt.manifest.json", encoding="utf-8"))
assert smoke_man["init_mode"] == "continued_pretrain", smoke_man["init_mode"]
assert smoke_man["parent_sha256"] == PARENT_SHA, "parent KHÔNG phải checkpoint đã gắn"
assert smoke["best_val_loss"] < 6.0, f"loss {smoke['best_val_loss']} quá cao: weights chưa được chuyển"
print("SMOKE OK | best_val_loss =", round(smoke["best_val_loss"], 4),
      "| step", smoke["iteration"], "| tokens", smoke["tokens_seen"])

## F. Pilot learning rate (nên chạy)

Cùng parent, cùng corpus, cùng seed; chọn theo độ ổn định + val sạch, không phải
train loss thấp nhất. 300 step mỗi mức.

In [ ]:
pilot = {}
for lr in ("3e-5", "1e-4", "3e-4"):
    run_out = f"{OUT}/lr_pilot_{lr}"
    subprocess.run([sys.executable, "train_gpt2.py",
                    "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                    "--init-from", CONFIG["checkpoint_path"], "--continue-pretrain",
                    "--out-dir", run_out, "--seq-len", "1024",
                    "--batch-size", "4", "--grad-accum", "8",
                    "--max-steps", "300", "--warmup", "20", "--lr", lr,
                    "--eval-interval", "100", "--eval-iters", "4",
                    "--save-interval", "100", "--session-max-minutes", "40",
                    "--seed", "17"]).check_returncode()
    blob = torch.load(f"{run_out}/latest.pt", map_location="cpu", weights_only=True, mmap=True)
    pilot[lr] = {"best_val_loss": blob["best_val_loss"], "step": blob["iteration"]}
print(json.dumps(pilot, indent=2))

## G. Continued pretraining — 100M token, một phiên

Global budget cố định: 4 × 8 × 1024 = 32,768 token/step → 3,052 step.
Không đổi số này ở các phiên sau. `--session-max-minutes` dừng an toàn, giữ nguyên
schedule và fingerprint.

In [ ]:
CHOSEN_LR = "1e-4"          # đổi theo kết quả Stage F
TOKENS_PER_STEP = 4 * 8 * 1024
TOTAL_STEPS = 100_000_000 // TOKENS_PER_STEP
print(TOTAL_STEPS, "steps x", TOKENS_PER_STEP, "=", TOTAL_STEPS * TOKENS_PER_STEP, "tokens")

subprocess.run([sys.executable, "train_gpt2.py",
                "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                "--init-from", CONFIG["checkpoint_path"], "--continue-pretrain",
                "--out-dir", f"{OUT}/run_100m", "--seq-len", "1024",
                "--batch-size", "4", "--grad-accum", "8",
                "--max-steps", str(TOTAL_STEPS), "--warmup", "100", "--lr", CHOSEN_LR,
                "--eval-interval", "250", "--eval-iters", "8",
                "--save-interval", "250", "--session-max-minutes", "500",
                "--seed", "17"]).check_returncode()

## G2. Phiên sau — resume đúng experiment

Chỉ `--resume-from` (không cần `--resume`). Nếu file không tồn tại, trainer **dừng**
chứ không train lại từ đầu. `--max-steps` phải giữ nguyên.

In [ ]:
PREV = CONFIG["phase17_ckpt_path"] or (glob.glob(
    "/kaggle/input/**/run_100m/latest.pt", recursive=True) + [None])[0]
if PREV and os.path.exists(PREV):
    subprocess.run([sys.executable, "train_gpt2.py",
                    "--preset", "gpt2_small_reference", "--data-dir", TRAIN_DATA,
                    "--resume-from", PREV, "--out-dir", f"{OUT}/run_100m",
                    "--seq-len", "1024", "--batch-size", "4", "--grad-accum", "8",
                    "--max-steps", str(TOTAL_STEPS), "--warmup", "100", "--lr", CHOSEN_LR,
                    "--eval-interval", "250", "--eval-iters", "8",
                    "--save-interval", "250", "--session-max-minutes", "500",
                    "--seed", "17"]).check_returncode()
else:
    print("Không có checkpoint phiên trước để resume — bỏ qua Stage G2.")

## H. Đánh giá + xuất kết quả

Task benchmark là file frozen của Phase 16. Language modelling chỉ đo khi có file
eval frozen; nếu không có thì ghi **NOT MEASURED**, không bịa số.

In [ ]:
CKPT = f"{OUT}/run_100m/best.pt"
assert os.path.exists(CKPT), "chưa có best.pt — Stage G chưa lưu checkpoint nào"

if CONFIG["eval_jsonl"] and os.path.exists(CONFIG["eval_jsonl"]):
    subprocess.run([sys.executable, "scripts/eval_phase16.py",
                    "--checkpoint", CKPT, "--val-jsonl", CONFIG["eval_jsonl"],
                    "--seq-len", "1024",
                    "--out", f"{OUT}/benchmark_results/eval_clean.json"]).check_returncode()
else:
    print("LANGUAGE MODELLING: NOT MEASURED (chưa gắn file eval frozen)")

subprocess.run([sys.executable, "scripts/phase16_task_benchmark.py",
                "--checkpoint", CKPT,
                "--out", f"{OUT}/benchmark_results/task_benchmark.json"]).check_returncode()

In [ ]:
import hashlib

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()

for name in ("latest.pt", "best.pt"):
    path = f"{OUT}/run_100m/{name}"
    if not os.path.exists(path):
        continue
    man = json.load(open(path + ".manifest.json", encoding="utf-8"))
    actual = sha(path)
    assert actual == man["sha256"], f"{name}: hash không khớp manifest"
    print(f"{name}: sha256={actual[:16]}… step={man['iteration']} "
          f"additional_tokens={man['additional_tokens_this_run']:,} "
          f"parent={man['parent_sha256'][:12]}… init_mode={man['init_mode']}")

print("\nSave Version để giữ output; phiên sau attach latest.pt rồi chạy Stage G2.")

## Recovery

* Không thấy `best.pt` / `*.parquet` / repo → Add Input dataset rồi chạy lại cell 0.
* `host_kind` không phải `kaggle` → bật Accelerator GPU và chạy trong Kaggle kernel.
* Smoke fail `loss ... quá cao` → weights chưa chuyển; kiểm tra `--init-from` và SHA parent.
* `resume requested but no checkpoint at ...` → attach checkpoint phiên trước.
* Hết disk giữa ingest → giảm `MAX_DOCS` (cell C2 tính lại theo dung lượng trống).
* Session dừng giữa chừng → `latest.pt` đã lưu atomic; chạy Stage G2 với cùng `TOTAL_STEPS`.